# Solution D2 · Calibration: who tells the truth when they say 30%?

**Type**: Interpret

## Objective

Compare five models on the same data for **discrimination** (AUC) and **calibration** (expected calibration error ECE, Brier score, log-loss), read the calibration curves
and verify that a post-hoc calibration (`CalibratedClassifierCV`) fixes a random forest.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(n=6000, with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income", "amount"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.5, stratify=df["defaulted"], random_state=0)
y_te = y_te.to_numpy()

models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "balanced": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "forest": RandomForestClassifier(300, random_state=0, n_jobs=-1),
    "forest_sigmoid": CalibratedClassifierCV(RandomForestClassifier(300, random_state=0, n_jobs=-1), method="sigmoid", cv=5),
    "forest_isotonic": CalibratedClassifierCV(RandomForestClassifier(300, random_state=0, n_jobs=-1), method="isotonic", cv=5),
}
probabilities = {name: m.fit(X_tr, y_tr).predict_proba(X_te)[:, 1] for name, m in models.items()}
print("frequency of positives in the test set:", y_te.mean().round(3))

## Commented solution

In [ ]:
def ece(y, p, k=10):
    d = pd.DataFrame({"y": y, "p": p, "g": pd.qcut(p, k, duplicates="drop")})
    s = d.groupby("g", observed=True).agg(n=("y", "size"), y=("y", "mean"), p=("p", "mean"))
    return float((s["n"] / s["n"].sum() * (s["p"] - s["y"]).abs()).sum())

In [ ]:
table = pd.DataFrame(index=list(probabilities), columns=["auc", "brier", "logloss", "ece", "mean_p"], dtype=float)
for name, p in probabilities.items():
    table.loc[name] = [roc_auc_score(y_te, p), brier_score_loss(y_te, p), log_loss(y_te, p), ece(y_te, p), p.mean()]
table.round(4)

In [ ]:
same_discrimination = bool(abs(table.loc["logistic", "auc"] - table.loc["balanced", "auc"]) < 0.005)
balanced_overestimation = table.loc["balanced", "mean_p"] - y_te.mean()
best_calibration = table["ece"].idxmin()
best_auc = table["auc"].idxmax()
isotonic_gain = table.loc["forest", "ece"] / table.loc["forest_isotonic", "ece"]
sigmoid_gain = table.loc["forest", "ece"] / table.loc["forest_sigmoid", "ece"]
why_balanced_is_skewed = "c"
print(f"balanced: equal AUC ({same_discrimination}), but mean probability {table.loc['balanced', 'mean_p']:.3f} against a true frequency of {y_te.mean():.3f}")
print(f"ECE of the forest: {table.loc['forest', 'ece']:.3f} -> sigmoid {table.loc['forest_sigmoid', 'ece']:.3f} (x{sigmoid_gain:.1f}) | isotonic {table.loc['forest_isotonic', 'ece']:.3f} (x{isotonic_gain:.1f})")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, p in probabilities.items():
    freq, mean = calibration_curve(y_te, p, n_bins=8, strategy="quantile")
    ax.plot(mean, freq, "o-", label=name, ms=4)
ax.plot([0, 1], [0, 1], "k:"); ax.set(xlabel="predicted probability", ylabel="observed frequency"); ax.legend(); plt.show()

## Check

In [ ]:
with verify():
    def reference(y, p, k=10):
        d = pd.DataFrame({"y": y, "p": p, "g": pd.qcut(p, k, duplicates="drop")}).groupby("g", observed=True).agg(n=("y", "size"), y=("y", "mean"), p=("p", "mean"))
        return float((d["n"] / d["n"].sum() * (d["p"] - d["y"]).abs()).sum())
    check(close(ece(y_te, probabilities["balanced"]), reference(y_te, probabilities["balanced"])) and close(ece(y_te, probabilities["forest"]), reference(y_te, probabilities["forest"])), "correct ECE", "groups of equal size with pd.qcut, then a weighted sum of the absolute gaps")
    check(table.notna().all().all() and close(table.loc["logistic", "auc"], roc_auc_score(y_te, probabilities["logistic"])), "complete table", "five rows and five columns")
    check(same_discrimination is True and balanced_overestimation > 0.2, "AUC practically equal, but `balanced` overestimates the frequency by over 0.2", "compare AUC and mean_p of the two models")
    check(best_calibration == "logistic" and best_auc in ("logistic", "balanced"), "the plain logistic is the best calibrated, and the two logistics beat the forests on AUC", "idxmin of the ECE, idxmax of the AUC")
    check(isotonic_gain > 2 and sigmoid_gain > 1.5, "calibration reduces the ECE of the forest by over 2 times (isotonic) and 1.5 times (sigmoid)", "ece(forest) / ece(calibrated version)")
    check(table.loc["forest_isotonic", "logloss"] < table.loc["forest", "logloss"] and table.loc["forest_isotonic", "brier"] < table.loc["forest", "brier"], "calibrating improves log-loss and Brier too", "look at the logloss and brier columns")
    check(why_balanced_is_skewed == "c", "correct answer: the weights shift the probabilities", "the AUC is almost equal, so the ordering is the same")
done()

## What you learned

- **Discrimination and calibration are different properties.** The logistic and its `balanced` version have the same AUC (0.764 against 0.763) but the second has mean probabilities of 0.43 when the true frequency is 0.19:
  the calibration curve sits clearly above the diagonal (ECE 0.24 against 0.01). For a decision with costs (B2) the model with weights is unusable without recalibrating.
- The random forest has a lower AUC (0.715) **and** less calibrated probabilities (ECE 0.059): the mean of the votes of the trees is not a probability. `CalibratedClassifierCV` fixes it (isotonic: ECE 0.018, sigmoid: 0.027) without changing
  the ordering, and improves log-loss and Brier too.
- The plain logistic is calibrated **by construction** (it maximizes a Bernoulli likelihood): if you need probabilities, it is an excellent starting point. If your model is a tree/forest/boosting,
  always check the calibration before reading "0.3" as "30%".
- Isotonic is more flexible than sigmoid but requires more data (with few examples it tends to overfit); sigmoid corrects only "S-shaped" distortions. With a few hundred examples prefer the sigmoid.

## Common pitfall

Choosing the model with the best AUC and using its probabilities as risk estimates. The AUC knows nothing about the scale of the probabilities: a model with AUC 0.92 can say "90%" for cases that
are positive 60 times out of 100.

## Interview question

<details><summary>Your model says 0.3: does it mean that 30% of these cases are positive?</summary>

Only if the model is calibrated. I verify it with a calibration curve (observed frequency against predicted probability for groups of cases) and an indicator like the ECE or the Brier score, on data not used for
training. The logistic is usually calibrated; forests, boosting and SVMs are not, and neither are models trained with weights or on rebalanced data: in that case I recalibrate (sigmoid or isotonic) with `CalibratedClassifierCV`.

</details>